# 🧹 Limpieza de datos con Python (pandas)
**Curso de IA · Bloque de Machine Learning · Sesión 1**

---

## ¿Por qué empezamos por aquí?

Hay una frase famosa en el mundo de los datos: **"Garbage in, garbage out"** (si entra basura, sale basura).
Un modelo de Machine Learning, por muy sofisticado que sea, **solo aprende de los datos que le damos**.
Si los datos tienen errores, el modelo aprenderá errores.

**Analogía:** cocinar. Antes de hacer una receta hay que lavar, pelar y cortar los ingredientes.
Limpiar datos es "lavar y pelar" antes de cocinar el modelo. En la práctica, es la parte que más tiempo ocupa en un proyecto real.

## ¿Qué vamos a hacer?

Trabajaremos con un CSV de **clientes de una tienda** que contiene los errores más típicos que te vas a encontrar en la vida real. Al final tendremos un CSV limpio, listo para entrenar modelos.

| # | Paso | Pregunta que nos hacemos |
|---|------|--------------------------|
| 1 | Cargar y mirar | ¿Qué tengo? ¿Cuántas filas y columnas? |
| 2 | Duplicados | ¿Hay filas repetidas? |
| 3 | Tipos de datos | ¿Los números son números y las fechas son fechas? |
| 4 | Texto inconsistente | ¿"Madrid" y "madrid " son lo mismo? |
| 5 | Valores imposibles y atípicos | ¿Hay edades de 250 años? |
| 6 | Valores nulos | ¿Qué hago con los huecos? |
| 7 | Categorías a números | ¿Cómo le explico "Madrid" a un modelo? |
| 8 | Escalado | ¿Edad (18-70) y salario (30.000) pesan lo mismo? |
| 9 | Guardar | Dejar el CSV limpio |

## Cómo usar este cuaderno

- Ejecuta cada celda de código con **Shift + Enter** (o el botón ▶).
- Hazlo **en orden**, de arriba abajo. Cada celda depende de las anteriores.
- Si algo se rompe: menú **Entorno de ejecución → Reiniciar y ejecutar todo**.
- Lee siempre el texto antes de ejecutar: primero entender, luego ejecutar.

> 👩‍🏫 **Guía de tiempos para el profesor** (orientativa, total ≈ 4 h con ejercicios)
>
> | Sección | Minutos |
> |---|---|
> | 1-2. Librerías y creación del CSV | 15 |
> | 3. Cargar y mirar | 25 |
> | 4. Duplicados | 15 |
> | 5. Tipos de datos | 25 |
> | 6. Texto inconsistente | 15 |
> | 7. Valores imposibles y outliers | 35 |
> | 8. Valores nulos | 35 |
> | 9. Categóricas a números | 25 |
> | 10. Escalado | 25 |
> | 11-12. Guardar, resumen y ejercicios | 25 |
>
> Sugerencia: pide a los alumnos que **predigan** el resultado antes de ejecutar cada celda. Genera mucha más discusión que ejecutar sin más.

## 📖 Mini glosario

| Término | Significado sencillo |
|---|---|
| **Dataset / DataFrame** | Una tabla de datos: filas = ejemplos, columnas = características |
| **Variable / característica (feature)** | Una columna (edad, salario...) |
| **Nulo / NaN** | Un hueco: dato que falta |
| **Outlier (atípico)** | Valor muy alejado de los demás (puede ser un error o algo real pero raro) |
| **Variable categórica** | Columna con categorías (ciudad, género) |
| **Escalado** | Poner varias columnas en una escala comparable |

## 1. Importar las librerías

Necesitamos tres "cajas de herramientas":

- **pandas** → trabajar con tablas (el Excel de Python)
- **numpy** → cálculos con números
- **matplotlib** → dibujar gráficos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Para que pandas nos muestre todas las columnas sin recortarlas
pd.set_option("display.max_columns", None)

print("Versión de pandas:", pd.__version__)

## 2. Crear nuestro CSV "sucio"

Para no depender de ningún fichero externo, la siguiente celda **fabrica un CSV con datos de clientes** y lo guarda como `clientes_sucio.csv`.

Cada fila es un cliente. La última columna, `compro`, indica si compró o no (más adelante será lo que queramos **predecir**).

Hemos metido a propósito estos **errores**, que iremos encontrando y arreglando uno a uno:

1. Filas **duplicadas**
2. Datos que **faltan** (nulos)
3. El salario está como **texto** (`"28.000 €"`)
4. La fecha está como **texto**
5. Ciudades escritas de **formas distintas** (`madrid`, `MADRID `, `Madrid`)
6. **Edades imposibles** (250 años, -5 años)
7. Un **salario exageradamente alto**

> 💡 No hace falta entender el código de esta celda: es la "fábrica de datos sucios".
> Más adelante podréis sustituir este paso por vuestro propio CSV.

In [ ]:
rng = np.random.default_rng(42)   # semilla: así todos obtenemos los mismos datos
n = 60

df_fabrica = pd.DataFrame({
    "id": range(1, n + 1),
    "nombre": [f"Cliente {i}" for i in range(1, n + 1)],
    "edad": rng.integers(18, 70, n).astype(float),
    "ciudad": rng.choice(["Madrid", "Barcelona", "Valencia", "Sevilla"], n),
    "salario": rng.normal(28000, 6000, n).round(0),
    "genero": rng.choice(["Hombre", "Mujer"], n),
    "fecha_alta": pd.date_range("2023-01-01", periods=n, freq="7D").strftime("%d/%m/%Y"),
    "compro": rng.choice(["Sí", "No"], n),
})

# --- Error 2: datos que faltan
df_fabrica.loc[[3, 10, 25, 40], "edad"] = np.nan
df_fabrica.loc[[5, 17, 33, 48, 52], "salario"] = np.nan
df_fabrica.loc[[8, 30], "ciudad"] = np.nan

# --- Error 6 y 7: valores imposibles / extremos
df_fabrica.loc[12, "edad"] = 250
df_fabrica.loc[20, "edad"] = -5
df_fabrica.loc[7, "salario"] = 250000

# --- Error 5: texto inconsistente
df_fabrica.loc[1, "ciudad"] = "madrid"
df_fabrica.loc[2, "ciudad"] = "  MADRID "
df_fabrica.loc[4, "ciudad"] = "Barcelona "

# --- Error 3: el salario pasa a ser texto con puntos y símbolo €
df_fabrica["salario"] = df_fabrica["salario"].map(
    lambda x: f"{x:,.0f} €".replace(",", ".") if pd.notna(x) else np.nan
)

# --- Error 1: filas duplicadas (copiamos las 3 primeras al final)
df_fabrica = pd.concat([df_fabrica, df_fabrica.iloc[[0, 1, 2]]], ignore_index=True)

df_fabrica.to_csv("clientes_sucio.csv", index=False, encoding="utf-8")
print("✅ Fichero 'clientes_sucio.csv' creado con", len(df_fabrica), "filas")

> 📁 **¿Quieres usar tu propio CSV?** En Colab, pulsa el icono de la carpeta 📁 (barra izquierda) y arrastra tu fichero, o ejecuta:
> ```python
> from google.colab import files
> files.upload()
> ```
> Luego cambia el nombre del fichero en `pd.read_csv(...)` del siguiente apartado.

## 3. Cargar el CSV y mirarlo (antes de tocar nada)

**Regla de oro:** antes de limpiar, hay que **mirar**. Un médico no receta sin explorar al paciente.

Usamos `pd.read_csv()` para leer el fichero. El resultado es un **DataFrame** (una tabla), que guardamos en la variable `df`.

In [ ]:
df = pd.read_csv("clientes_sucio.csv")

# head(10) muestra las 10 primeras filas
df.head(10)

### 3.1 ¿Cuántas filas y columnas tengo?
`shape` devuelve (filas, columnas).

In [ ]:
print("Filas y columnas:", df.shape)

### 3.2 Tipos de datos y huecos con `info()`

Fíjate en dos cosas:

- **Non-Null Count**: cuántos valores NO vacíos hay en cada columna. Si es menor que el total de filas, hay **nulos**.
- **Dtype**: el tipo de dato. `object` significa **texto**. `float64` es número decimal, `int64` es entero.

👀 ¿Ves algo raro? `salario` debería ser un número y `fecha_alta` una fecha, pero ambas aparecen como `object` (texto).

In [ ]:
df.info()

### 3.3 Resumen estadístico con `describe()`

Para las columnas numéricas nos da: media, desviación, mínimo, máximo y cuartiles.

👀 Mira `min` y `max` de la **edad**. ¿Tienen sentido?
Y fíjate que **`salario` no aparece**: pandas la trata como texto, no como número. Es una pista más de que hay que convertirla.

In [ ]:
df.describe()

In [ ]:
# Para las columnas de texto: cuántos valores distintos hay y cuál es el más frecuente
df.select_dtypes(exclude="number").describe()

### 3.4 Guardamos una copia de seguridad

`.copy()` crea una copia independiente. Así, si nos equivocamos, podemos comparar con el original.

In [ ]:
df_original = df.copy()

## 4. Filas duplicadas

A veces un mismo registro entra dos veces (un formulario enviado dos veces, una fusión de ficheros...).
Si no lo quitamos, el modelo "cree" que ese cliente es más importante que los demás.

- `df.duplicated()` marca con `True` las filas que repiten una anterior.
- `.sum()` cuenta cuántas hay (True = 1).

In [ ]:
print("Filas duplicadas:", df.duplicated().sum())

# keep=False muestra TODAS las copias (la original y las repetidas) para verlas juntas
df[df.duplicated(keep=False)].sort_values("id")

Las eliminamos con `drop_duplicates()`.
`reset_index(drop=True)` renumera las filas de 0 en adelante para que no queden "huecos" en el índice.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("Filas ahora:", df.shape[0])

## 5. Arreglar los tipos de datos

Un número guardado como texto **no se puede sumar, ni promediar, ni meter en un modelo**.

### 5.1 El salario: de texto (`"28.000 €"`) a número (`28000.0`)

Hay que quitar el símbolo `€`, quitar el punto de los miles y convertir a número.
Encadenamos varias operaciones de texto con `.str`:

1. `.str.replace("€", "")` → quita el símbolo
2. `.str.replace(".", "")` → quita el punto de los miles
3. `.str.strip()` → quita espacios sobrantes
4. `.astype(float)` → convierte a número

In [ ]:
print("ANTES:")
print(df["salario"].head(5).tolist())

df["salario"] = (
    df["salario"]
    .str.replace("€", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.strip()
    .astype(float)
)

print("\nDESPUÉS:")
print(df["salario"].head(5).tolist())

### 5.2 La fecha: de texto a fecha real

Con `pd.to_datetime` indicamos el **formato** en que está escrita (`%d/%m/%Y` = día/mes/año).
Una vez es fecha de verdad, podemos extraer partes como el mes con `.dt.month`.

In [ ]:
df["fecha_alta"] = pd.to_datetime(df["fecha_alta"], format="%d/%m/%Y")
df["mes_alta"] = df["fecha_alta"].dt.month   # nueva columna con el mes (1-12)

df[["fecha_alta", "mes_alta"]].head()

In [ ]:
# Comprobamos que los tipos ya son correctos
df.dtypes

## 6. Texto inconsistente

Para una persona, `Madrid`, `madrid` y `  MADRID ` son la misma ciudad.
**Para el ordenador son tres categorías distintas.** Veámoslo con `value_counts()`, que cuenta cuántas veces aparece cada valor.
`dropna=False` hace que cuente también los nulos.

In [ ]:
df["ciudad"].value_counts(dropna=False)

La solución estándar: quitar espacios (`.str.strip()`) y unificar mayúsculas (`.str.title()` → "Madrid").
Lo aplicamos a todas las columnas de texto de categorías.

In [ ]:
for col in ["ciudad", "genero", "compro"]:
    df[col] = df[col].str.strip().str.title()

print(df["ciudad"].value_counts(dropna=False))
print()
print("Valores de genero:", df["genero"].unique())
print("Valores de compro:", df["compro"].unique())

> 💡 Los nulos (`NaN`) no se ven afectados por `.str`: siguen siendo nulos. Los trataremos más adelante.

## 7. Valores imposibles y valores atípicos (outliers)

Hay dos situaciones distintas:

- **Valor imposible**: no puede ser real (edad = 250). Es un error seguro.
- **Valor atípico (outlier)**: es muy distinto del resto, pero *podría* ser real (un salario de 250.000 €). Hay que **decidir** con criterio.

### 7.1 Primero, mirarlos: el diagrama de caja (boxplot)

La caja contiene el 50 % central de los datos. Los puntos sueltos fuera de los "bigotes" son valores sospechosos.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 4))

ejes[0].boxplot(df["edad"].dropna())
ejes[0].set_title("Edad (antes de limpiar)")

ejes[1].boxplot(df["salario"].dropna())
ejes[1].set_title("Salario (antes de limpiar)")

plt.tight_layout()
plt.show()

👀 En la edad se ven valores muy lejos (250 y -5). En el salario hay un punto muy alto que "aplasta" el resto del gráfico.

### 7.2 Valores imposibles: aplicar una regla de sentido común

Decidimos que una edad válida está entre **16 y 100 años**. Lo que quede fuera lo marcamos como nulo (`np.nan`) para tratarlo después junto con los demás huecos.

Esto se llama una **regla de negocio**: la pone una persona que conoce el problema, no el ordenador.

In [ ]:
# Filas con edad imposible
imposibles = df[(df["edad"] < 16) | (df["edad"] > 100)]
print("Edades imposibles encontradas:")
print(imposibles[["id", "edad"]])

# Las convertimos en nulo
df.loc[(df["edad"] < 16) | (df["edad"] > 100), "edad"] = np.nan

### 7.3 Valores atípicos: el método del rango intercuartílico (IQR)

Un método muy usado para detectar outliers de forma objetiva:

1. **Q1** = valor por debajo del cual está el 25 % de los datos
2. **Q3** = valor por debajo del cual está el 75 % de los datos
3. **IQR = Q3 − Q1** (la "anchura" de la caja)
4. Se consideran atípicos los valores por debajo de **Q1 − 1,5·IQR** o por encima de **Q3 + 1,5·IQR**

Es exactamente lo que dibuja el boxplot con esos puntos sueltos.

In [ ]:
Q1 = df["salario"].quantile(0.25)
Q3 = df["salario"].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Q1 = {Q1:.0f} | Q3 = {Q3:.0f} | IQR = {IQR:.0f}")
print(f"Salarios válidos entre {limite_inferior:.0f} y {limite_superior:.0f}")

atipicos = df[(df["salario"] < limite_inferior) | (df["salario"] > limite_superior)]
print("\nSalarios atípicos encontrados:")
print(atipicos[["id", "salario"]])

### ¿Qué hacemos con un outlier? Es una decisión, no una fórmula

| Opción | Cuándo tiene sentido |
|---|---|
| **Eliminar la fila** | Es claramente un error y sobran datos |
| **Convertir en nulo y rellenar** | Es un error pero no queremos perder el resto de la fila |
| **Dejarlo** | Es real e importante (ej. detectar fraudes: el outlier **es** lo que buscamos) |
| **Limitarlo (`clip`)** | Es real pero demasiado extremo; lo "recortamos" al límite |

En nuestro caso suponemos que es un error de registro, así que lo **convertimos en nulo**.

In [ ]:
df.loc[(df["salario"] < limite_inferior) | (df["salario"] > limite_superior), "salario"] = np.nan

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
ejes[0].boxplot(df["edad"].dropna())
ejes[0].set_title("Edad (después)")
ejes[1].boxplot(df["salario"].dropna())
ejes[1].set_title("Salario (después)")
plt.tight_layout()
plt.show()

## 8. Valores nulos (datos que faltan)

Casi ningún modelo de scikit-learn acepta nulos: da error. Hay que decidir qué hacer.

### 8.1 Detectar: ¿cuántos hay y dónde?

`df.isna()` marca con True los huecos. `.sum()` los cuenta por columna. `.mean()` da la proporción (× 100 = porcentaje).

In [ ]:
nulos = df.isna().sum()
porcentaje = (df.isna().mean() * 100).round(1)

pd.DataFrame({"nulos": nulos, "% del total": porcentaje})

In [ ]:
# Gráfico de barras de los nulos por columna
nulos[nulos > 0].plot(kind="bar", color="tomato", figsize=(6, 3))
plt.title("Valores nulos por columna")
plt.ylabel("Cantidad")
plt.xticks(rotation=0)
plt.show()

### 8.2 Dos grandes estrategias

| Estrategia | Cómo | Ventaja | Inconveniente |
|---|---|---|---|
| **Eliminar** filas con nulos | `df.dropna()` | Simple, no inventamos datos | **Perdemos información** |
| **Rellenar** (imputar) | `df.fillna(valor)` | Conservamos todas las filas | Estamos "estimando" |

Veamos cuánto perderíamos si eliminamos:

In [ ]:
print("Filas actuales:", len(df))
print("Filas que quedarían con dropna():", len(df.dropna()))
print("Filas perdidas:", len(df) - len(df.dropna()))

Con pocas filas, perder datos duele. Así que **rellenamos**:

- **Numéricas → mediana** (el valor central). Mejor que la media cuando hay valores extremos, porque la media se deja arrastrar por ellos.
- **Categóricas → moda** (el valor más frecuente).

> 🧠 Otras opciones más avanzadas: media, rellenar con un valor fijo, `KNNImputer`, `SimpleImputer` de scikit-learn... Las veremos más adelante.

In [ ]:
# Calculamos cada valor de relleno y lo mostramos
mediana_edad = df["edad"].median()
mediana_salario = df["salario"].median()
moda_ciudad = df["ciudad"].mode()[0]     # mode() devuelve una lista; [0] = el más frecuente

print("Mediana de edad:", mediana_edad)
print("Mediana de salario:", mediana_salario)
print("Ciudad más frecuente:", moda_ciudad)

# Rellenamos
df["edad"] = df["edad"].fillna(mediana_edad)
df["salario"] = df["salario"].fillna(mediana_salario)
df["ciudad"] = df["ciudad"].fillna(moda_ciudad)

print("\nNulos que quedan:")
print(df.isna().sum())

> 👩‍🏫 **Nota para el profesor (para sesiones posteriores):** en un proyecto real, las medianas y modas se calculan **solo con el conjunto de entrenamiento** y luego se aplican al de prueba, para no "filtrar" información del test al entrenamiento (*data leakage*). Aquí lo hacemos sobre todo el dataset por simplicidad didáctica; cuando veamos `train_test_split` y `Pipeline` lo haremos correctamente.

## 9. Convertir categorías en números

Los modelos solo entienden **números**. `"Madrid"` o `"Mujer"` no significan nada para ellos. Hay que traducir.

### 9.1 Quitar columnas que no sirven

- `id` y `nombre`: identifican al cliente pero **no aportan para predecir** (si las dejamos, el modelo podría "memorizar" en vez de aprender).
- `fecha_alta`: ya hemos extraído lo útil (`mes_alta`); un modelo no entiende fechas directamente.

In [ ]:
df = df.drop(columns=["id", "nombre", "fecha_alta"])
df.head()

### 9.2 Categorías de dos valores → 0 y 1 (`map`)

Para `genero` y `compro` basta con asignar un número a cada valor con un diccionario.

In [ ]:
df["genero"] = df["genero"].map({"Hombre": 0, "Mujer": 1})
df["compro"] = df["compro"].map({"No": 0, "Sí": 1})

df[["genero", "compro"]].head()

### 9.3 Categorías con varios valores → *One-Hot Encoding* (`get_dummies`)

¿Por qué no poner Madrid = 1, Barcelona = 2, Valencia = 3...?
Porque el modelo pensaría que **Valencia (3) es "más" que Madrid (1)**, o que Barcelona es "la media" entre las dos. Ese orden no existe.

**Solución:** una columna nueva por ciudad, con 1 si el cliente es de esa ciudad y 0 si no.

| ciudad | → | ciudad_Madrid | ciudad_Barcelona | ciudad_Valencia | ciudad_Sevilla |
|---|---|---|---|---|---|
| Madrid | → | 1 | 0 | 0 | 0 |
| Valencia | → | 0 | 0 | 1 | 0 |

In [ ]:
df = pd.get_dummies(df, columns=["ciudad"], dtype=int)
df.head()

## 10. Escalado de variables

**Problema:** `edad` va de 18 a 70, `salario` ronda los 28.000. Para algunos modelos (kNN, SVM, KMeans, regresiones con regularización) que miden **distancias**, el salario "gritaría" mucho más fuerte que la edad solo por tener números más grandes.

**Solución:** poner las columnas en una escala comparable. Dos técnicas muy comunes:

| Técnica | Qué hace | Resultado |
|---|---|---|
| **StandardScaler** (estandarización) | Resta la media y divide por la desviación | Media ≈ 0, desviación ≈ 1 |
| **MinMaxScaler** (normalización) | Lleva todo al rango 0-1 | Mínimo = 0, máximo = 1 |

> 🌳 Los modelos basados en **árboles** (Decision Tree, Random Forest) **no necesitan** escalado.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

columnas = ["edad", "salario"]

df_estandar = df.copy()
df_estandar[columnas] = StandardScaler().fit_transform(df[columnas])

df_minmax = df.copy()
df_minmax[columnas] = MinMaxScaler().fit_transform(df[columnas])

print("ORIGINAL"); print(df[columnas].describe().loc[["mean", "std", "min", "max"]].round(2))
print("\nESTANDARIZADO"); print(df_estandar[columnas].describe().loc[["mean", "std", "min", "max"]].round(2))
print("\nMIN-MAX"); print(df_minmax[columnas].describe().loc[["mean", "std", "min", "max"]].round(2))

👀 Fíjate: en el estandarizado la media sale ≈ 0 y la desviación ≈ 1; en el min-max el mínimo es 0 y el máximo 1.

Veámoslo dibujado. **La forma del histograma no cambia; solo cambia la escala del eje horizontal.**

In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(12, 6))

for fila, col in enumerate(columnas):
    ejes[fila, 0].hist(df[col], bins=10, color="steelblue")
    ejes[fila, 0].set_title(f"{col} - original")
    ejes[fila, 1].hist(df_estandar[col], bins=10, color="seagreen")
    ejes[fila, 1].set_title(f"{col} - estandarizado")
    ejes[fila, 2].hist(df_minmax[col], bins=10, color="darkorange")
    ejes[fila, 2].set_title(f"{col} - min-max")

plt.tight_layout()
plt.show()

> 👩‍🏫 **Nota para el profesor:** guardaremos el CSV **sin escalar**. El escalado es mejor hacerlo justo antes de entrenar, **ajustándolo solo con los datos de entrenamiento** (dentro de un `Pipeline`). Lo veremos en la sesión de modelos.

## 11. Comprobación final y guardado

Comparemos el dataset original con el limpio.

In [ ]:
print("ORIGINAL :", df_original.shape, "| nulos totales:", int(df_original.isna().sum().sum()))
print("LIMPIO   :", df.shape, "| nulos totales:", int(df.isna().sum().sum()))
print()
df.info()

In [ ]:
df.head(10)

Todas las columnas son ya **numéricas** y no quedan nulos: este dataset **ya se puede usar para entrenar un modelo**.

Guardamos el resultado con `to_csv`. `index=False` evita que se guarde la columna del índice (0, 1, 2...).

In [ ]:
df.to_csv("clientes_limpio.csv", index=False)
print("✅ Guardado: clientes_limpio.csv")

# En Colab, esto descarga el fichero a tu ordenador (si no estás en Colab, no hace nada)
try:
    from google.colab import files
    files.download("clientes_limpio.csv")
except ImportError:
    pass

> ⚠️ Los ficheros que creas en Colab **se borran** cuando se cierra la sesión. Descárgalos o guárdalos en Google Drive si quieres conservarlos.

## 12. Resumen: la "lista de comprobación" de limpieza

| Paso | Pregunta | Herramienta en pandas |
|---|---|---|
| 1. Mirar | ¿Qué tengo? | `head()`, `shape`, `info()`, `describe()` |
| 2. Duplicados | ¿Filas repetidas? | `duplicated()`, `drop_duplicates()` |
| 3. Tipos | ¿Cada columna tiene el tipo correcto? | `astype()`, `pd.to_datetime()` |
| 4. Texto | ¿Mismas categorías escritas distinto? | `.str.strip()`, `.str.title()`, `value_counts()` |
| 5. Imposibles y outliers | ¿Valores absurdos o extremos? | `boxplot`, IQR, reglas de negocio |
| 6. Nulos | ¿Datos que faltan? | `isna()`, `dropna()`, `fillna()` |
| 7. Categorías | ¿Texto que el modelo no entiende? | `map()`, `get_dummies()` |
| 8. Escalado | ¿Columnas con escalas muy distintas? | `StandardScaler`, `MinMaxScaler` |

**Idea clave para recordar:** limpiar no es aplicar una receta mecánica. En cada paso hay una **decisión** (¿elimino o relleno? ¿es un error o un dato real?) y conviene **dejar por escrito el porqué**.

## ✍️ Ejercicios para los alumnos

**Nivel 1 – Repaso**
1. ¿Cuántas filas se habrían perdido si hubiéramos hecho `dropna()` justo después de cargar el CSV (sin limpiar nada más)? *(Pista: usa `df_original`.)*
2. ¿Qué ciudad era la más frecuente? ¿Qué pasaría si rellenáramos los nulos de `ciudad` con otra?

**Nivel 2 – Practicar**
3. Rellena los nulos de `edad` con la **media** en vez de la mediana. ¿Cambia mucho el resultado?
4. Crea una columna nueva `tramo_edad` con `pd.cut` (joven: menos de 30, adulto: 30-50, senior: más de 50).
5. Cambia el límite de edad válido a 18-65. ¿Cuántas filas se ven afectadas?

**Nivel 3 – Reto**
6. Escribe una **función** `limpiar_clientes(ruta_csv)` que haga todo el proceso y devuelva el DataFrame limpio.
7. Descarga otro CSV (por ejemplo, el del Titanic) y aplica la lista de comprobación de la sección 12. Anota qué problemas encuentras y qué decisiones tomas.

---
### 🎯 Próxima sesión
Con un dataset limpio ya podemos hacer nuestro **primer modelo de Machine Learning**: separar entrenamiento y prueba, entrenar, predecir y medir.